# US Refinery Explorer — end-to-end

Single notebook that:

1. Applies the `refineries.*` schema migration (idempotent — safe to re-run).
2. Lists every US refinery in `oil_network.nodes`.
3. Lets you pick **`calibration`** (3 hand-picked refineries) or **`full`** (all 115) via a single variable.
4. Runs the Claude Agent SDK research loop on each target.
5. Persists results to `refineries.*` + `outputs/refineries/<refinery_id>/profile.json`.
6. Sanity-checks the row counts for one calibration refinery.

## Prerequisites (one-off)

- `..\..\.venv\Scripts\pip.exe install claude-agent-sdk` — already done; `0.2.87` is in the venv.
- `claude login` — make sure the Claude Code CLI is authenticated with your Max subscription. The Agent SDK picks up the subscription credit automatically; no `ANTHROPIC_API_KEY` is read.

## Cost model

Your Claude Code Max plan includes a **$100/month Agent SDK credit**. Calibration (3 refineries) should cost a few dollars; the full 115-refinery sweep is the variable to watch. After credit exhaustion the SDK falls back to API billing — so run calibration first and read the per-refinery `cost_usd` before unleashing on all 115.

In [1]:
# %% Setup
import sys, json, time
from pathlib import Path

# code/ is the working dir; make sure it's importable.
CODE_DIR = Path.cwd()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import psycopg2

from us_refinery_explorer import (
    list_us_refineries,
    explore_refinery_async,
    persist_to_db,
    already_explored,
    REFINERIES_OUT,
)

REFINERIES_OUT.mkdir(parents=True, exist_ok=True)
print('output dir:', REFINERIES_OUT)

output dir: C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Stage2\outputs\refineries


In [2]:
# %% Apply the schema migration (idempotent)
#
# CREATE SCHEMA IF NOT EXISTS + CREATE TABLE IF NOT EXISTS — safe to re-run.
# Calls the canonical migration script so the notebook and the standalone CLI
# stay in lock-step.

from migrations.create_refineries_schema import main as apply_refineries_schema
apply_refineries_schema()

refineries schema created (or already existed).
  tables present: ['events', 'exploration_runs', 'financials', 'process_units', 'refinery', 'runs_monthly', 'slate', 'sources']


In [3]:
# %% Load refineries from the DB
refineries = list_us_refineries()
print(f'US refineries in DB: {len(refineries)}')
print()
print(f'{"refinery_id":40} {"PADD":5} {"capacity":>10}  name')
for r in refineries[:10]:
    cap = f"{r['capacity_bpd']:,}" if r['capacity_bpd'] else '?'
    print(f'{r["refinery_id"]:40} {(r["padd"] or "-"):5} {cap:>10}  {r["name"][:60]}')
print(f'... ({len(refineries) - 10} more)')

US refineries in DB: 115

refinery_id                              PADD    capacity  name
ref_pbf_delaware_city                    PADD1    171,000  DELAWARE CITY REFINING CO LLC — DELAWARE CITY
ref_cpi_paulsboro                        PADD1     32,000  CPI OPERATIONS LLC — PAULSBORO
ref_paulsboro_paulsboro                  PADD1    160,000  PAULSBORO REFINING CO LLC — PAULSBORO
ref_p66_bayway                           PADD1    258,500  PHILLIPS 66 COMPANY — LINDEN
ref_american_bradford                    PADD1     11,000  AMERICAN REFINING GROUP INC — BRADFORD
ref_monroe_trainer                       PADD1    190,000  MONROE ENERGY LLC — TRAINER
ref_united_warren                        PADD1     67,000  UNITED REFINING CO — WARREN
ref_ergon_newell                         PADD1     22,300  ERGON WEST VIRGINIA INC — NEWELL
ref_exxon_joliet                         PADD2    264,000  EXXONMOBIL REFINING & SUPPLY CO — JOLIET
ref_marathon_robinson                    PADD2    253,000  MARATHO

In [4]:
# %% Configure the run — EDIT THESE VARIABLES
#
# MODE:  'calibration' → 3 hand-picked refineries (start here)
#        'full'        → all 115
#        'custom'      → use the explicit CUSTOM_IDS list below
#
# MODEL: model identifier passed to ClaudeAgentOptions. Defaults to Sonnet 4.6.
#
# FORCE: if False, refineries with a successful exploration_runs row are
#        skipped (resume-aware). Set True to re-run.

MODE  = 'custom'              # 'calibration' | 'full' | 'custom'
MODEL = 'claude-sonnet-4-6'
FORCE = True

CALIBRATION_IDS = [
    'ref_p66_bayway',           # Phillips 66 — Linden NJ, 258 kbpd, FCC + coker, lots of public data
    'ref_pbf_delaware_city',    # PBF Energy  — Delaware City, 190 kbpd, FCC + coker
    'ref_american_bradford',    # American Refining Group — Bradford PA, small, sparse public data
]

CUSTOM_IDS = [
    'ref_american_bradford',    # smoke test: single small refinery, fixed EIA tool
]

if MODE == 'calibration':
    chosen_ids = CALIBRATION_IDS
elif MODE == 'custom':
    chosen_ids = CUSTOM_IDS
elif MODE == 'full':
    chosen_ids = [r['refinery_id'] for r in refineries]
else:
    raise ValueError(f'unknown MODE: {MODE!r}')

targets = [r for r in refineries if r['refinery_id'] in set(chosen_ids)]
missing = set(chosen_ids) - {r['refinery_id'] for r in targets}
if missing:
    print(f'WARNING: requested IDs not in DB: {missing}')

print(f'mode={MODE!r}  model={MODEL!r}  force={FORCE}  targets={len(targets)}')
for r in targets[:20]:
    cap = f"{r['capacity_bpd']:,}" if r['capacity_bpd'] else '?'
    print(f'  {r["refinery_id"]:40} {cap:>10}  {r["name"][:55]}')
if len(targets) > 20:
    print(f'  ... ({len(targets) - 20} more)')

mode='custom'  model='claude-sonnet-4-6'  force=True  targets=1
  ref_american_bradford                        11,000  AMERICAN REFINING GROUP INC — BRADFORD


In [5]:
# %% Run the agent across targets
#
# Top-level `await` works in Jupyter / IPython. Each refinery is one full
# agent run; results are persisted to DB + profile.json before moving on, so
# a kernel interrupt mid-sweep loses at most one refinery's work.

summary = []
for i, r in enumerate(targets, 1):
    rid = r['refinery_id']
    if not FORCE and already_explored(rid):
        print(f'[{i}/{len(targets)}] skipped (already explored): {rid}')
        continue

    print(f'[{i}/{len(targets)}] exploring {rid}: {r["name"]}')
    t0 = time.time()
    try:
        buf, meta = await explore_refinery_async(r, model=MODEL)
        persist_to_db(buf, r, meta)
        elapsed = time.time() - t0
        print(f'    status={meta["status"]}  tool_calls={meta["tool_calls"]}  '
              f'cost=${meta["cost_usd"]:.3f}  elapsed={elapsed:.0f}s  '
              f'units={len(buf["units"])} slate={len(buf["slate"])} '
              f'events={len(buf["events"])} fin={len(buf["financials"])} '
              f'monthly={len(buf["monthly"])} sources={len(buf["sources"])}')
        if buf['summary']:
            print(f'    summary: {buf["summary"][:200]}')
        summary.append({**meta, 'refinery_id': rid, 'name': r['name']})
    except Exception as exc:
        elapsed = time.time() - t0
        print(f'    FAILED after {elapsed:.0f}s: {exc!r}')
        summary.append({'refinery_id': rid, 'name': r['name'], 'status': 'failed', 'error': repr(exc)})

print()
print('done. summary:')
for s in summary:
    print(f'  {s.get("refinery_id"):40} {s.get("status")}  '
          f'tool_calls={s.get("tool_calls")}  cost=${s.get("cost_usd", 0):.3f}')

[1/1] exploring ref_american_bradford: AMERICAN REFINING GROUP INC — BRADFORD


    status=failed  tool_calls=0  cost=$0.000  elapsed=0s  units=0 slate=0 events=0 fin=0 monthly=0 sources=0

done. summary:
  ref_american_bradford                    failed  tool_calls=0  cost=$0.000


In [6]:
# %% Inspect the outputs for one refinery (calibration sanity-check)
#
# Edit INSPECT_ID to look at any refinery that's been explored.

INSPECT_ID = (CALIBRATION_IDS[0] if MODE == 'calibration'
              else (chosen_ids[0] if chosen_ids else None))

if INSPECT_ID is None:
    print('nothing to inspect')
else:
    print(f'inspecting refinery: {INSPECT_ID}')
    conn = psycopg2.connect(host='localhost', dbname='eia_crude',
                            user='eia_user', password='eia_password')
    try:
        for table in ('refinery', 'process_units', 'slate', 'events',
                      'financials', 'runs_monthly', 'sources', 'exploration_runs'):
            cur = conn.cursor()
            cur.execute(f'SELECT COUNT(*) FROM refineries.{table} '
                        f'WHERE refinery_id = %s', (INSPECT_ID,))
            n = cur.fetchone()[0]
            print(f'  refineries.{table:20} rows: {n}')
            cur.close()
    finally:
        conn.close()

    profile_path = REFINERIES_OUT / INSPECT_ID / 'profile.json'
    if profile_path.exists():
        profile = json.loads(profile_path.read_text())
        print(f'\nprofile.json at {profile_path}:')
        print(f'  summary       : {profile["result"]["summary"]}')
        print(f'  units         : {len(profile["result"]["units"])}')
        print(f'  slate entries : {len(profile["result"]["slate"])}')
        print(f'  events        : {len(profile["result"]["events"])}')
        print(f'  financials    : {len(profile["result"]["financials"])}')
        print(f'  monthly pts   : {len(profile["result"]["monthly"])}')
        print(f'  sources       : {len(profile["result"]["sources"])}')
    else:
        print(f'\nno profile.json yet at {profile_path}')

inspecting refinery: ref_american_bradford


  refineries.refinery             rows: 1
  refineries.process_units        rows: 5
  refineries.slate                rows: 3
  refineries.events               rows: 11
  refineries.financials           rows: 1
  refineries.runs_monthly         rows: 86
  refineries.sources              rows: 11
  refineries.exploration_runs     rows: 2

profile.json at C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Stage2\outputs\refineries\ref_american_bradford\profile.json:
  summary       : None
  units         : 0
  slate entries : 0
  events        : 0
  financials    : 0
  monthly pts   : 0
  sources       : 0
